In [13]:
import numpy as np
import pandas as pd
import tables_io

In [3]:
## 5-sigma limiting depths ------------------
### LSST: median values for COSMOS deep field from https://usdf-maf.slac.stanford.edu/summaryStats?runId=5#Basics_Coadd%20M5
### Roman from https://github.com/jfcrenshaw/photerr/blob/a014b39729ddde3daf80be2dbe82f5a7f958882c/photerr/roman.py#L120-L126
### HSC Niji (60 min exposure) from https://sites.google.com/view/hsc-mb-survey3/filter-specification?authuser=0 
###
### Acessed June 3, 2026
### --------------------------------------------

M5_DEPTHS_DeepField = {'LSST_u'    : 27.74,
                      'LSST_g'    : 28.69,
                      'LSST_r'    : 28.88,
                      'LSST_i'    : 28.96,
                      'LSST_z'    : 28.26,
                      'LSST_y'    : 26.63,
                      'Roman_F062': 27.7,
                      'Roman_F087': 27.7,
                      'Roman_F106': 27.6,
                      'Roman_F129': 27.5,
                      'Roman_F158': 27.0,
                      'Roman_F184': 25.9,
                      'Roman_F213': 28.3,
                      'HSC_MB_00' : 26.41,
                      'HSC_MB_01' : 26.51,
                      'HSC_MB_02' : 26.45,
                      'HSC_MB_03' : 26.69,
                      'HSC_MB_04' : 26.93,
                      'HSC_MB_05' : 26.62,
                      'HSC_MB_06' : 26.26,
                      'HSC_MB_07' : 26.02,
                      'HSC_MB_08' : 26.07,
                      'HSC_MB_09' : 26.00,
                      'HSC_MB_10' : 26.06,
                      'HSC_MB_11' : 25.52,
                      'HSC_MB_12' : 25.58,
                      'HSC_MB_13' : 25.43,
                      'HSC_MB_14' : 25.15,
                      'HSC_MB_15' : 24.79}


### 5-sigma limiting depths for WideFastDeep from https://usdf-maf.slac.stanford.edu/summaryStats?runId=5#Basics_Coadd%20M5
### Column `DD:WFD CoaddM5`
M5_DEPTHS_WideFastDeep = {'LSST_u'    : 25.61,
                          'LSST_g'    : 26.90,
                          'LSST_r'    : 26.87,
                          'LSST_i'    : 26.43,
                          'LSST_z'    : 25.73,
                          'LSST_y'    : 24.79}

### Get list of bands
BANDS_DeepField = list(M5_DEPTHS_DeepField.keys())
ERR_BANDS_DeepField = [f"{key}_err" for key in BANDS_DeepField]

BANDS_WideFastDeep = list(M5_DEPTHS_WideFastDeep.keys())
ERR_BANDS_WideFastDeep = [f"{key}_err" for key in BANDS_WideFastDeep]

In [28]:
## Prepare input data
TRAINING_DATA_UMAP = tables_io.read("/pscratch/sd/s/sajkov/analysis_pipeline/runs/06Jul26_11kDataPoints/PHOTOMETRY_DeepField_noisy_06Jul26.pq")

TRAINING_PHOTOMETRY_UMAP = TRAINING_DATA_UMAP[BANDS_WideFastDeep]
TRAINING_PHOTOMERRS_UMAP = TRAINING_DATA_UMAP[ERR_BANDS_WideFastDeep]

TRAINING_REDSHIFTS_UMAP_specZs  = np.load("/pscratch/sd/s/sajkov/analysis_pipeline/runs/06Jul26_11kDataPoints/TRUEREDSHIFTS_DeepField.pkl.npy")
TRAINING_REDSHIFTS_UMAP_photoZs = tables_io.read("/pscratch/sd/s/sajkov/analysis_pipeline/runs/06Jul26_11kDataPoints/PHOTOZS_DeepField_lePhare_06Jul26.pq")

ESTIMATION_DATA_UMAP = tables_io.read("/pscratch/sd/s/sajkov/analysis_pipeline/runs/06Jul26_11kDataPoints/PHOTOMETRY_WideFastDeep_noisy_06Jul26.pq")

ESTIMATION_PHOTOMETRY_UMAP = ESTIMATION_DATA_UMAP[BANDS_WideFastDeep]
ESTIMATION_PHOTOMERRS_UMAP = ESTIMATION_DATA_UMAP[ERR_BANDS_WideFastDeep]

column_list None
column_list None
column_list None


In [29]:
COLORNAMES_WideFastDeep = [f"{BANDS_WideFastDeep[i].split('_')[-1]}-{BANDS_WideFastDeep[i + 1].split('_')[-1]}"
                           for i in range(len(BANDS_WideFastDeep) - 1)]

COLERRNAMES_WideFastDeep = [f"{BANDS_WideFastDeep[i].split('_')[-1]}-{BANDS_WideFastDeep[i + 1].split('_')[-1]}_err"
                           for i in range(len(BANDS_WideFastDeep) - 1)]

In [47]:
COLORS_WideFastDeep = pd.DataFrame(
    {COLORNAMES_WideFastDeep[i]:
        TRAINING_PHOTOMETRY_UMAP[BANDS_WideFastDeep[i]] - TRAINING_PHOTOMETRY_UMAP[BANDS_WideFastDeep[i + 1]]
            for i in range(len(BANDS_WideFastDeep) - 1)}
)

# COLERRS_WideFastDeep = pd.DataFrame(
#     {COLERRNAMES_WideFastDeep[i]:
#         TRAINING_PHOTOMERRS_UMAP[ERR_BANDS_WideFastDeep[i]] - TRAINING_PHOTOMERRS_UMAP[ERR_BANDS_WideFastDeep[i + 1]]
#             for i in range(len(ERR_BANDS_WideFastDeep) - 1)}
# )

COLERRS_WideFastDeep = pd.DataFrame(
    {COLERRNAMES_WideFastDeep[i]:
        np.clip(np.sqrt(TRAINING_PHOTOMERRS_UMAP[ERR_BANDS_WideFastDeep[i]]**2 + TRAINING_PHOTOMERRS_UMAP[ERR_BANDS_WideFastDeep[i + 1]]**2),
                None, 0.05)
            for i in range(len(ERR_BANDS_WideFastDeep) - 1)}
)

In [40]:
TRAINING_PHOTOMETRY_UMAP

,LSST_u,LSST_g,LSST_r,LSST_i,LSST_z,LSST_y
index,,,,,,
232136,26.074474,26.139794,26.039279,25.495192,25.809098,25.241489
1184099,29.242006,29.423577,28.832412,26.593863,25.712719,25.517956
1373785,25.302653,24.198437,23.728439,23.506351,23.385119,23.284056
1767073,27.138766,26.436428,26.018765,25.906271,25.858277,25.910881
566949,24.247786,23.970521,23.360148,22.874242,22.713242,22.628279
...,...,...,...,...,...,...
1155869,28.957846,28.637337,26.698958,26.328271,26.320241,26.196137
419882,25.887895,25.445968,25.063193,24.554449,24.221012,23.757473
894501,29.220125,28.595875,27.978868,28.023723,27.782204,27.332550


In [48]:
TRAINING_PHOTOMERRS_UMAP

,LSST_u_err,LSST_g_err,LSST_r_err,LSST_i_err,LSST_z_err,LSST_y_err
index,,,,,,
232136,0.047011,0.021322,0.016635,0.010239,0.023255,0.060462
1184099,0.529180,0.369643,0.200539,0.025057,0.021377,0.077730
1373785,0.023533,0.006095,0.005356,0.005212,0.005572,0.011151
1767073,0.123281,0.027686,0.016352,0.013967,0.024282,0.110901
566949,0.010045,0.005746,0.005189,0.005075,0.005180,0.007401
...,...,...,...,...,...,...
1155869,0.484063,0.199670,0.029536,0.019869,0.036681,0.143123
419882,0.039702,0.012035,0.008173,0.006262,0.007266,0.016199
894501,0.526717,0.192690,0.094104,0.091158,0.137644,0.361922


In [50]:
COLORS_WideFastDeep

,u-g,g-r,r-i,i-z,z-y
index,,,,,
232136,-0.065319,0.100515,0.544087,-0.313906,0.567609
1184099,-0.181571,0.591164,2.238550,0.881143,0.194763
1373785,1.104216,0.469998,0.222088,0.121232,0.101063
1767073,0.702338,0.417663,0.112494,0.047994,-0.052604
566949,0.277265,0.610373,0.485906,0.160999,0.084964
...,...,...,...,...,...
1155869,0.320509,1.938379,0.370686,0.008031,0.124104
419882,0.441927,0.382775,0.508744,0.333437,0.463539
894501,0.624250,0.617007,-0.044855,0.241519,0.449655


In [51]:
COLERRS_WideFastDeep

,u-g_err,g-r_err,r-i_err,i-z_err,z-y_err
index,,,,,
232136,0.050000,0.027044,0.019533,0.025409,0.050000
1184099,0.050000,0.050000,0.050000,0.032936,0.050000
1373785,0.024310,0.008113,0.007473,0.007630,0.012466
1767073,0.050000,0.032155,0.021505,0.028013,0.050000
566949,0.011573,0.007742,0.007258,0.007252,0.009034
...,...,...,...,...,...
1155869,0.050000,0.050000,0.035597,0.041716,0.050000
419882,0.041486,0.014548,0.010296,0.009592,0.017754
894501,0.050000,0.050000,0.050000,0.050000,0.050000


In [9]:
COLORS_WideFastDeep

['u-g', 'g-r', 'r-i', 'i-z', 'z-y']

In [7]:
TRAINING_PHOTOMETRY_UMAP_MAGNITUDES

,LSST_u,LSST_g,LSST_r,LSST_i,LSST_z,LSST_y
index,,,,,,
232136,26.074474,26.139794,26.039279,25.495192,25.809098,25.241489
1184099,29.242006,29.423577,28.832412,26.593863,25.712719,25.517956
1373785,25.302653,24.198437,23.728439,23.506351,23.385119,23.284056
1767073,27.138766,26.436428,26.018765,25.906271,25.858277,25.910881
566949,24.247786,23.970521,23.360148,22.874242,22.713242,22.628279
...,...,...,...,...,...,...
1155869,28.957846,28.637337,26.698958,26.328271,26.320241,26.196137
419882,25.887895,25.445968,25.063193,24.554449,24.221012,23.757473
894501,29.220125,28.595875,27.978868,28.023723,27.782204,27.332550
